# Teste das funções de gráfico

Demonstração de `src/graficos.py` com **dados fictícios** (gerados aleatoriamente). Nenhum número aqui vem do ENEM; os textos são marcadores de posição.

In [ ]:
import numpy as np
import pandas as pd

from src import graficos as gr

gr.aplicar_tema()
rng = np.random.default_rng(42)
REDES = ["federal", "estadual_municipal", "privada"]

## Paletas

In [ ]:
print("redes:", gr.CORES_REDE)
print("sequencial (5):", gr.cores_sequenciais(5))
print("destaque:", gr.DESTAQUE)

## `distribuicoes`

In [ ]:
fict = pd.DataFrame({
    "rede": rng.choice(REDES, size=6000, p=[0.1, 0.6, 0.3]),
})
fict["nota"] = fict["rede"].map({"federal": 600, "estadual_municipal": 500, "privada": 580}) + rng.normal(0, 70, len(fict))

fig, ax = gr.distribuicoes(
    fict, "nota", "rede", mostrar_n=True,
    titulo="Título de exemplo", subtitulo="Subtítulo de exemplo (dados fictícios)",
    rotulo_x="Rótulo do eixo x", rotulo_y="Densidade", nota="Nota de rodapé de exemplo.",
)

In [ ]:
fig, ax = gr.distribuicoes(
    fict, "nota", "rede", tipo="histograma", bins=30,
    titulo="Título de exemplo (histograma)", rotulo_x="Rótulo do eixo x",
)

## `serie_temporal`

In [ ]:
anos = np.arange(2020, 2026)
serie = pd.DataFrame([
    {"ano": a, "rede": r, "media": base + rng.normal(0, 6) + (-15 if a >= 2024 else 0)}
    for r, base in zip(REDES, [610, 505, 590]) for a in anos
])

fig, ax = gr.serie_temporal(
    serie, "ano", "media", "rede", marcar_quebra=True, rotulo_quebra="rótulo da quebra",
    titulo="Título de exemplo", subtitulo="Subtítulo de exemplo (dados fictícios)",
    rotulo_y="Rótulo do eixo y",
)

Sem ligar as linhas através da quebra, e com um grupo em destaque (ocre):

In [ ]:
muni = pd.DataFrame([
    {"ano": a, "municipio": m, "taxa": base + rng.normal(0, 0.02)}
    for m, base in zip(["município A", "município B", "município C"], [0.55, 0.48, 0.62]) for a in anos
])

fig, ax = gr.serie_temporal(
    muni, "ano", "taxa", "municipio", destaque="município A",
    marcar_quebra=True, conectar_quebra=False,
    titulo="Título de exemplo (destaque)",
)

## `barras_com_n`

In [ ]:
faixas = ["faixa 1", "faixa 2", "faixa 3", "faixa 4", "faixa 5"]
barras = pd.DataFrame({
    "faixa": faixas,
    "media": [470, 495, 520, 560, 610] + rng.normal(0, 5, 5),
    "qtd": [42000, 31000, 15500, 6200, 1800],
})

fig, ax = gr.barras_com_n(
    barras, "faixa", "media", "qtd", sequencial=True, casas_valor=0,
    titulo="Título de exemplo (faixas ordenadas)", rotulo_y="Rótulo do eixo y",
)

In [ ]:
agrup = pd.DataFrame([
    {"faixa": f, "rede": r, "taxa": rng.uniform(20, 80), "qtd": int(rng.integers(50, 20000))}
    for f in faixas[:3] for r in REDES
])

fig, ax = gr.barras_com_n(
    agrup, "faixa", "taxa", "qtd", coluna_grupo="rede", sufixo_valor="%",
    titulo="Título de exemplo (agrupadas)",
)

In [ ]:
fig, ax = gr.barras_com_n(
    barras, "faixa", "media", "qtd", horizontal=True, destaque="faixa 3", casas_valor=0,
    titulo="Título de exemplo (horizontal, destaque)",
)

Categoria `nao_sei` em faixas ordenadas: cinza, fora da sequência, depois dos níveis.

In [ ]:
niveis = ["nivel_1", "nivel_2", "nivel_3", "nivel_4", "nao_sei"]
esc = pd.DataFrame({
    "escolaridade": niveis,
    "media": [480, 510, 545, 590, 470] + rng.normal(0, 5, 5),
    "qtd": [21000, 18000, 12500, 7300, 2600],
})

fig, ax = gr.barras_com_n(
    esc, "escolaridade", "media", "qtd", sequencial=True, casas_valor=0,
    titulo="Título de exemplo (nao_sei)",
)

In [ ]:
esc_ind = pd.DataFrame({"escolaridade": rng.choice(niveis, size=5000)})
esc_ind["nota"] = esc_ind["escolaridade"].map(dict(zip(niveis, [480, 510, 545, 590, 470]))) + rng.normal(0, 60, len(esc_ind))

fig, ax = gr.distribuicoes(
    esc_ind, "nota", "escolaridade", sequencial=True, ordem=niveis,
    titulo="Título de exemplo (densidades, nao_sei)",
)

## `nota` e `legenda`

`nota` aparece abaixo do eixo x; `legenda=False` oculta a legenda.

In [ ]:
fig, ax = gr.distribuicoes(
    fict, "nota", "rede",
    titulo="Título de exemplo (nota)", rotulo_x="Rótulo do eixo x",
    nota="Nota de exemplo.",
)

In [ ]:
fig, ax = gr.serie_temporal(
    serie, "ano", "media", "rede", legenda=False,
    titulo="Título de exemplo (nota, sem legenda)", rotulo_x="Rótulo do eixo x",
    nota="Nota de exemplo.",
)

In [ ]:
fig, ax = gr.barras_com_n(
    agrup, "faixa", "taxa", "qtd", coluna_grupo="rede", sufixo_valor="%",
    titulo="Título de exemplo (nota)",
    nota="Nota de exemplo.",
)

## `mapa_municipios`

Valores fictícios sobre os códigos reais da malha do IBGE (`data/geo/mg_municipios_2025.geojson`, gerada por `scripts/geo_mg.py`). Parte dos municípios fica sem valor de propósito, para mostrar a cor neutra.

In [ ]:
malha = gr.carregar_malha()
mapa = malha.sample(frac=0.9, random_state=42).copy()
mapa["nota"] = rng.normal(480, 25, len(mapa))
mapa["qtd"] = rng.integers(5, 3000, len(mapa))

# Continuous scale over the data range: no arrows on the bar.
fig, ax = gr.mapa_municipios(
    mapa, "nota", "codigo_municipio",
    titulo="Título de exemplo", subtitulo="Subtítulo de exemplo (dados fictícios)",
    rotulo_barra="Nota média em CN (pontos)", rotulo_sem_dados="rótulo sem dados",
    nota="Nota de exemplo.",
)

Com `n_minimo` (municípios com `qtd` abaixo de 100 viram "sem dados"), classes definidas por `cortes` e um município em `destaque` (contorno ocre):

In [ ]:
# Discrete classes (faixas=): values outside [440, 520] take the end
# classes and the bar gets arrows on both ends.
fig, ax = gr.mapa_municipios(
    mapa, "nota", "codigo_municipio", coluna_n="qtd", n_minimo=100,
    faixas=[440, 460, 480, 500, 520],
    destaque=3106200,
    titulo="Título de exemplo (n mínimo, faixas, destaque)",
    rotulo_barra="Nota média em CN (pontos)", rotulo_sem_dados="rótulo sem dados",
)

Painel de mapas (ex.: dois anos lado a lado): mesmos `vmin`/`vmax` nos dois mapas para usarem a mesma escala de cores; `painel` deixa a barra de cores só no primeiro.

In [ ]:
import matplotlib.pyplot as plt

# vmin/vmax shared by both maps and narrower than the data: arrows on both ends.
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, ano in zip(axes, [2024, 2025]):
    mapa_ano = malha.sample(frac=0.9, random_state=ano).copy()
    mapa_ano["nota"] = rng.normal(480, 25, len(mapa_ano))
    mapa_ano["qtd"] = rng.integers(5, 3000, len(mapa_ano))
    gr.mapa_municipios(
        mapa_ano, "nota", "codigo_municipio", coluna_n="qtd", n_minimo=100,
        vmin=440, vmax=520, destaque=3106200,
        rotulo_barra="Nota média em CN (pontos)", rotulo_sem_dados="rótulo sem dados",
        titulo=str(ano), ax=ax,
    )
gr.painel(fig, axes, titulo="Título de exemplo (painel de mapas)",
          subtitulo="Subtítulo de exemplo (dados fictícios)", nota="Nota de exemplo.")

## `funil`

Unidades fictícias: `n` com distribuição log-normal e média sorteada em torno de uma referência com erro `dp / sqrt(n)`; algumas unidades recebem um deslocamento extra, para haver pontos fora das curvas. `media_ref` e `dp_ref` são passados por quem chama.

In [ ]:
MEDIA_REF, DP_REF = 500, 80

def unidades_ficticias(k=850, deslocamento=0.0):
    n = np.round(rng.lognormal(mean=4, sigma=1.2, size=k)).astype(int) + 1
    media = MEDIA_REF + deslocamento + rng.normal(0, DP_REF / np.sqrt(n))
    extra = rng.random(k) < 0.04
    media[extra] += rng.choice([-1, 1], extra.sum()) * rng.uniform(25, 60, extra.sum())
    return pd.DataFrame({"unidade": [f"unidade {i:03d}" for i in range(k)], "qtd": n, "media": media})

funil_df = unidades_ficticias()

fig, ax = gr.funil(
    funil_df, "qtd", "media", media_ref=MEDIA_REF, dp_ref=DP_REF,
    titulo="Título de exemplo", subtitulo="Subtítulo de exemplo (dados fictícios)",
    rotulo_x="Rótulo do eixo x (escala log)", rotulo_y="Rótulo do eixo y",
    nota="Nota de exemplo.",
)

`limites_funil` também serve para contar pontos fora das curvas:

In [ ]:
inf, sup = gr.limites_funil(funil_df["qtd"], MEDIA_REF, DP_REF, z=3)
fora = (funil_df["media"] < inf) | (funil_df["media"] > sup)
print(f"{fora.sum()} de {len(funil_df)} unidades fora dos limites com z=3")

Com `destaque` (ocre, por cima, com rótulo), `rotular` (rótulo sem mudar a cor) e `n_minimo=5`:

In [ ]:
maior = funil_df.nlargest(1, "qtd")["unidade"].iloc[0]
fora_lista = funil_df.loc[fora & (funil_df["qtd"] >= 5), "unidade"].head(2).tolist()

fig, ax = gr.funil(
    funil_df, "qtd", "media", media_ref=MEDIA_REF, dp_ref=DP_REF, n_minimo=5,
    coluna_rotulo="unidade", destaque=maior, rotular=fora_lista,
    titulo="Título de exemplo (destaque, rotular, n mínimo)",
    rotulo_x="Rótulo do eixo x", rotulo_y="Rótulo do eixo y",
)

## `painel`

Painel 2x3 de `funil`, um ano por painel, eixos compartilhados. `painel` aplica título, subtítulo e nota da figura, põe os rótulos dos eixos só nas bordas e deixa a legenda só no primeiro painel. Chamar por último, antes de `salvar`.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(12, 7.5), sharex=True, sharey=True)
for ax, ano in zip(axes.flat, range(2020, 2026)):
    gr.funil(
        unidades_ficticias(deslocamento=rng.normal(0, 4)), "qtd", "media",
        media_ref=MEDIA_REF, dp_ref=DP_REF, titulo=str(ano), ax=ax,
    )
gr.painel(
    fig, axes,
    titulo="Título de exemplo (painel)", subtitulo="Subtítulo de exemplo (dados fictícios)",
    nota="Nota de exemplo.\nSegunda linha da nota.",
    rotulo_x="Rótulo do eixo x (escala log)", rotulo_y="Rótulo do eixo y",
)

Com uma célula vazia: o painel acima dela passa a ser borda (rótulo e números do eixo x). A legenda dos outros painéis fica oculta, não removida, então os títulos dos painéis ficam na mesma altura.

In [ ]:
fig2, axes2 = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
for ax, titulo in zip(axes2.flat, ["painel 1", "painel 2", "painel 3"]):
    gr.serie_temporal(serie, "ano", "media", "rede", titulo=titulo, ax=ax)
axes2[1, 1].remove()
gr.painel(fig2, axes2, titulo="Título de exemplo", nota="Nota de exemplo.",
          rotulo_x="Rótulo x", rotulo_y="Rótulo y")

## `salvar`

In [ ]:
caminho = gr.salvar(fig, "99_teste_graficos")
print(caminho.relative_to(gr.ROOT))